In [7]:
import pandas as pd

departments = pd.read_csv("../data/raw/departments.csv")
print("Initial shape:", departments.shape)
departments.head(20)

Initial shape: (20, 2)


,Department_ID,Department_Name
0,D001,Cardiology
1,D002,Neurology
2,D003,Orthopedics
3,D004,Pediatrics
4,D005,Oncology
5,D006,ENT
6,D007,Dermatology
7,D008,General Surgery
8,D009,Urology
9,D010,Nephrology


In [8]:
print("Missing values:\n", departments.isnull().sum())
print("\nDuplicate rows:", departments.duplicated().sum())
print("Duplicate Department_ID:", departments['Department_ID'].duplicated().sum())
print("\nData types:\n", departments.dtypes)

Missing values:
 Department_ID      0
Department_Name    0
dtype: int64

Duplicate rows: 0
Duplicate Department_ID: 0

Data types:
 Department_ID      str
Department_Name    str
dtype: object


In [9]:
df = departments.copy()

# Since we only have 2 known text columns, no need for select_dtypes
# (avoids the Pandas4Warning entirely)
df['Department_ID'] = df['Department_ID'].str.strip()
df['Department_Name'] = df['Department_Name'].str.strip()

# Title-case names, but preserve known acronyms
df['Department_Name'] = df['Department_Name'].str.title()

acronym_fixes = {
    'Ent': 'ENT',
    'Icu': 'ICU'
}
df['Department_Name'] = df['Department_Name'].replace(acronym_fixes)

# Validate ID format
invalid_ids = df[~df['Department_ID'].str.match(r'^D\d{3}$')]
print("Invalid ID formats:\n", invalid_ids)

# Rename to snake_case
df = df.rename(columns={
    'Department_ID': 'department_id',
    'Department_Name': 'department_name'
})

df = df.sort_values('department_id').reset_index(drop=True)
df.head(20)

Invalid ID formats:
 Empty DataFrame
Columns: [Department_ID, Department_Name]
Index: []


,department_id,department_name
0,D001,Cardiology
1,D002,Neurology
2,D003,Orthopedics
3,D004,Pediatrics
4,D005,Oncology
5,D006,ENT
6,D007,Dermatology
7,D008,General Surgery
8,D009,Urology
9,D010,Nephrology


In [10]:
# Sanity checks
assert 'Ent' not in df['department_name'].values
assert 'Icu' not in df['department_name'].values
assert df['department_name'].str.contains('ENT|ICU', regex=True).sum() == 2
assert df['department_id'].is_unique
assert df.isnull().sum().sum() == 0
print("All sanity checks passed.")

All sanity checks passed.


In [11]:
df.to_csv("../data/processed/departments_clean.csv", index=False)
print("Saved:", df.shape)

Saved: (20, 2)
